In [1]:
# Import necessary libraries
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "amex-46887"
DATASET_ID = "amex_risk"

client = bigquery.Client(project=PROJECT_ID)

In [2]:
# List tables in the dataset
query = f"""
SELECT table_name
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

client.query(query).to_dataframe()

,table_name
0,customer_features_base
1,customer_features_latest
2,customer_features_v2
3,model_dataset_v1
4,sample_submission_raw
5,test_data_raw
6,train_data_raw
7,train_joined
8,train_labels_raw


In [3]:
# Check the number of rows in the train_data_raw table
query = f"""
SELECT COUNT(*) AS n_rows
FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
"""
client.query(query).to_dataframe()

,n_rows
0,5531451


In [4]:
# Unique Customers
query = f"""
SELECT COUNT(DISTINCT customer_ID) AS unique_customers
FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
"""
client.query(query).to_dataframe()

,unique_customers
0,458913


In [5]:
# Label Balance
query = f"""
SELECT target, COUNT(*) AS n_customers
FROM `{PROJECT_ID}.{DATASET_ID}.train_labels_raw`
GROUP BY target
ORDER BY target
"""
client.query(query).to_dataframe()

,target,n_customers
0,0,340085
1,1,118828


In [6]:
# Missing label customers
query = f"""
SELECT COUNT(*) AS missing_label_customers
FROM `{PROJECT_ID}.{DATASET_ID}.train_labels_raw` l
LEFT JOIN (
  SELECT DISTINCT customer_ID
  FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
) t
USING (customer_ID)
WHERE t.customer_ID IS NULL
"""
client.query(query).to_dataframe()

,missing_label_customers
0,0


In [7]:
# Distribution of number of statements per customer
query = f"""
WITH statement_counts AS (
  SELECT customer_ID, COUNT(*) AS n_statements
  FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
  GROUP BY customer_ID
)
SELECT n_statements, COUNT(*) AS n_customers
FROM statement_counts
GROUP BY n_statements
ORDER BY n_statements
"""
client.query(query).to_dataframe()

,n_statements,n_customers
0,1,5120
1,2,6098
2,3,5778
3,4,4673
4,5,4671
5,6,5515
6,7,5198
7,8,6110
8,9,6411
9,10,6721


In [8]:
# Statements per customer statistics
query = f"""
WITH statement_counts AS (
  SELECT customer_ID, COUNT(*) AS n_statements
  FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
  GROUP BY customer_ID
)
SELECT
  MIN(n_statements) AS min_statements,
  MAX(n_statements) AS max_statements,
  AVG(n_statements) AS avg_statements
FROM statement_counts
"""
client.query(query).to_dataframe()

,min_statements,max_statements,avg_statements
0,1,13,12.053376


## Observations

- `train_data_raw` contains 5,531,451 rows and 458,913 unique customers.
- `train_labels_raw` contains 458,913 labelled customers.
- The target is imbalanced, with more non-default cases than default cases.
- There are no labelled customers missing from the raw transaction table.
- Customers have between 1 and 13 monthly statements, with an average of about 12.05.
- This confirms the dataset has a customer-level sequential structure suitable for aggregated and temporal feature engineering.

In [9]:
from pathlib import Path

sql_path = Path("../sql/04_latest_record_features.sql")
query = sql_path.read_text()

client.query(query).result()
print("customer_features_latest created successfully.")

customer_features_latest created successfully.


In [10]:
query = """
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT customer_ID) AS customer_count
FROM `amex-46887.amex_risk.customer_features_base`
"""
client.query(query).to_dataframe()

,row_count,customer_count
0,458913,458913


In [11]:
query = """
SELECT *
FROM `amex-46887.amex_risk.customer_features_base`
LIMIT 5
"""
client.query(query).to_dataframe()

,customer_ID,target,n_statements,P_2_avg,P_2_min,P_2_max,P_2_std,P_2_nulls,D_39_avg,D_39_min,...,B_11_avg,B_11_min,B_11_max,B_11_std,B_11_nulls,S_3_avg,S_3_min,S_3_max,S_3_std,S_3_nulls
0,1d632c583e759354dbd17dbafc7451f45a343a4db549a0...,1,1,0.491013,0.491013,0.491013,NaN,0,0.001021,0.001021,...,0.027008,0.027008,0.027008,NaN,0,0.407605,0.407605,0.407605,NaN,0
1,806016c0b3f6f05f67dcb0a54874e647446e78e72ea6a7...,1,1,0.535045,0.535045,0.535045,NaN,0,0.005590,0.005590,...,0.035395,0.035395,0.035395,NaN,0,0.361063,0.361063,0.361063,NaN,0
2,9ea06f2f8e6979c093aa7e659270684acad22360fef72b...,1,1,NaN,NaN,NaN,NaN,1,0.003965,0.003965,...,1.280536,1.280536,1.280536,NaN,0,0.787184,0.787184,0.787184,NaN,0
3,019d0a289e59bf806655be8d5e6954fa46e7f06b2b31c8...,0,1,0.574732,0.574732,0.574732,NaN,0,0.004864,0.004864,...,0.145471,0.145471,0.145471,NaN,0,0.251044,0.251044,0.251044,NaN,0
4,12a282ab15f26e3390e094d23b85e902fc901175f28487...,1,1,0.471331,0.471331,0.471331,NaN,0,0.651788,0.651788,...,0.317947,0.317947,0.317947,NaN,0,0.220528,0.220528,0.220528,NaN,0


In [12]:
from pathlib import Path

query = Path("../sql/03_base_customer_features.sql").read_text()
client.query(query).result()
print("customer_features_base recreated successfully.")

customer_features_base recreated successfully.


In [13]:
query = """
SELECT *
FROM `amex-46887.amex_risk.customer_features_base`
LIMIT 1
"""
df_check = client.query(query).to_dataframe()
df_check.columns.tolist()

['customer_ID',
 'target',
 'n_statements',
 'P_2_avg',
 'P_2_min',
 'P_2_max',
 'P_2_std',
 'D_39_avg',
 'D_39_min',
 'D_39_max',
 'D_39_std',
 'B_9_avg',
 'B_9_min',
 'B_9_max',
 'B_9_std',
 'R_1_avg',
 'R_1_min',
 'R_1_max',
 'R_1_std',
 'B_11_avg',
 'B_11_min',
 'B_11_max',
 'B_11_std',
 'S_3_avg',
 'S_3_min',
 'S_3_max',
 'S_3_std']

In [14]:
from pathlib import Path

for sql_file in [
    "../sql/04_latest_record_features.sql",
    "../sql/05_model_dataset.sql",
]:
    query = Path(sql_file).read_text()
    client.query(query).result()
    print(f"Finished: {sql_file}")

Finished: ../sql/04_latest_record_features.sql
Finished: ../sql/05_model_dataset.sql


In [15]:
query = """
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT customer_ID) AS customer_count
FROM `amex-46887.amex_risk.model_dataset_v1`
"""
client.query(query).to_dataframe()

,row_count,customer_count
0,458913,458913


In [16]:
query = """
SELECT *
FROM `amex-46887.amex_risk.model_dataset_v1`
LIMIT 5
"""
client.query(query).to_dataframe()

,customer_ID,target,n_statements,P_2_avg,P_2_min,P_2_max,P_2_std,D_39_avg,D_39_min,D_39_max,...,S_3_min,S_3_max,S_3_std,latest_statement_date,P_2_latest,D_39_latest,B_9_latest,R_1_latest,B_11_latest,S_3_latest
0,e5bf002290207350f23b70d371424e10df4094f32929c7...,0,2,0.948683,0.945841,0.951525,0.004019,0.357201,0.030504,0.683898,...,0.199385,0.199385,NaN,2018-03-01,0.951525,0.683898,0.019208,0.000149,0.039413,0.199385
1,024866a2bee469c2048c1e6710d586539b0dcaa4af009b...,0,2,0.634114,0.633778,0.634451,0.000476,0.080943,0.066496,0.095391,...,0.601799,0.614677,0.009106,2018-03-01,0.633778,0.095391,0.006369,0.002458,0.116294,0.614677
2,ca13b43a6dcee32f321c15013aece393d51d66b029d50e...,0,5,0.593923,0.574205,0.626738,0.021165,0.072421,0.005906,0.183576,...,0.052927,0.163350,0.050601,2018-03-01,0.626738,0.183576,0.165613,0.006319,0.007207,0.052927
3,31cf39c8e91ff0065b4c8c76866aa80aaff36d8e7de3f6...,0,5,0.529533,0.507219,0.543991,0.016447,0.316607,0.009838,0.620303,...,0.149404,0.205139,0.022710,2018-03-01,0.543991,0.208890,0.008723,0.001374,0.078600,0.155892
4,68ad102f92c5f9f3c21911ce5f166c6c656e2eb692e895...,0,6,0.428348,0.376419,0.465147,0.045771,0.200410,0.000014,0.392001,...,0.101250,0.129517,0.010845,2018-03-01,0.458475,0.000014,0.151517,0.000842,0.000502,0.103458


In [17]:
query = """
SELECT
  target,
  COUNT(*) AS n_customers
FROM `amex-46887.amex_risk.model_dataset_v1`
GROUP BY target
ORDER BY target
"""
client.query(query).to_dataframe()

,target,n_customers
0,0,340085
1,1,118828


# ============================================
# Validate model_dataset_v2
# ============================================

In [19]:
from google.cloud import bigquery

PROJECT_ID = "amex-46887"
BQ_LOCATION = "US"

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

In [21]:
query = """
SELECT table_name
FROM `amex-46887.amex_risk.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""
client.query(query).to_dataframe()

,table_name
0,customer_features_base
1,customer_features_latest
2,customer_features_v2
3,model_dataset_v1
4,sample_submission_raw
5,test_data_raw
6,train_data_raw
7,train_joined
8,train_labels_raw


In [23]:
query = """
SELECT *
FROM `amex-46887.amex_risk.model_dataset_v2`
LIMIT 5
"""
df_preview = client.query(query).to_dataframe()
df_preview.head()

,customer_ID,target,n_statements,P_2_avg,P_2_min,P_2_max,P_2_std,P_2_range,D_39_avg,D_39_min,...,D_39_missing_pct,B_9_missing_count,B_9_missing_pct,R_1_missing_count,R_1_missing_pct,B_11_missing_count,B_11_missing_pct,S_3_missing_count,S_3_missing_pct,history_days
0,0ce494a08251a7493cfdab4b4348d903dd88c9908b1287...,0,13,0.899208,0.867373,0.915915,0.014855,0.048542,0.130881,0.000144,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,389
1,138e0604d8e5ad253cd8c049cb3e84fe3b27915889282a...,0,13,0.831651,0.805928,0.867648,0.019745,0.061720,0.007104,0.001337,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,366
2,9189a3926509c9a08a48e3894bf042ec9e55fe6c99a615...,0,13,0.902865,0.864215,0.978155,0.031982,0.113939,0.255876,0.005088,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,373
3,d384a51f830e5bc55ff365715c1cb462dea5670dbba4c0...,1,13,0.438565,0.295611,0.512445,0.071107,0.216834,0.060003,0.001054,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,381
4,ed2e5d27e894398391847ef13752bcb7ba14a674498112...,0,13,0.723871,0.666130,0.802647,0.049746,0.136517,0.282936,0.001385,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,368


In [24]:
query = """
SELECT COUNT(*) AS n_rows
FROM `amex-46887.amex_risk.model_dataset_v2`
"""
client.query(query).to_dataframe()

,n_rows
0,458913


In [25]:
query = """
SELECT column_name
FROM `amex-46887.amex_risk.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'model_dataset_v2'
ORDER BY ordinal_position
"""
client.query(query).to_dataframe()

,column_name
0,customer_ID
1,target
2,n_statements
3,P_2_avg
4,P_2_min
...,...
61,B_11_missing_count
62,B_11_missing_pct
63,S_3_missing_count
64,S_3_missing_pct


In [26]:
query = """
SELECT
  COUNTIF(P_2_avg IS NULL) AS P_2_avg_nulls,
  COUNTIF(P_2_latest IS NULL) AS P_2_latest_nulls,
  COUNTIF(P_2_first IS NULL) AS P_2_first_nulls
FROM `amex-46887.amex_risk.model_dataset_v2`
"""
client.query(query).to_dataframe()

,P_2_avg_nulls,P_2_latest_nulls,P_2_first_nulls
0,2434,2969,17498


In [27]:
query = """
SELECT target, COUNT(*) AS n
FROM `amex-46887.amex_risk.model_dataset_v2`
GROUP BY target
ORDER BY target
"""
client.query(query).to_dataframe()

,target,n
0,0,340085
1,1,118828
